In [112]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path


In [113]:
conda_dir = Path("data/")
if conda_dir.exists() and conda_dir.is_dir():
    csv_files = sorted(conda_dir.glob("*.csv"))
else:
    csv_files = sorted(Path(".").glob("*CONDA*.csv"))

conda_data = {file.stem: pd.read_csv(file) for file in csv_files}

print(f"Loaded {len(conda_data)} CSV file(s).")
for name, df in conda_data.items():
    print(f"{name}: {df.shape}")

Loaded 3 CSV file(s).
CONDA_test: (8974, 7)
CONDA_train: (26921, 10)
CONDA_valid: (8974, 10)


In [114]:
print("Removing Rows...")
for row in ["Id", "conversationId", "playerId", "matchId", "playerSlot", "slotClasses", "slotTokens"]:
    print(f"{'Removed:':<15}{row:>15}")

for name, df in conda_data.items():
    if name != "CONDA_test":
        drop_candidates = {"Id", "conversationId", "playerId", "matchId", "playerSlot", "slotClasses", "slotTokens"}
        ordered_drop_cols = [col for col in df.columns if col in drop_candidates]
        df.drop(columns=ordered_drop_cols, inplace=True)
    else:
        drop_candidates = {"Id", "conversationId", "playerId", "matchId", "playerSlot", "slotTokens"}
        ordered_drop_cols = [col for col in df.columns if col in drop_candidates]
        df.drop(columns=ordered_drop_cols, inplace=True)

print()
for name, df in conda_data.items():
    print(f"{name}: {df.shape}")

output_dir = conda_dir / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

Removing Rows...
Removed:                    Id
Removed:        conversationId
Removed:              playerId
Removed:               matchId
Removed:            playerSlot
Removed:           slotClasses
Removed:            slotTokens

CONDA_test: (8974, 2)
CONDA_train: (26921, 3)
CONDA_valid: (8974, 3)


In [115]:
print("Normalizing game times...")

for name, df in conda_data.items():
    print(f"{name} chatTime: {df['chatTime'].sample(n=1, random_state=42).iloc[0]} -> ", end='')

    floor=-90 
    ceiling=3600

    clipped = df["chatTime"].clip(lower=floor, upper=ceiling)
    df["chatTime"] = round((clipped - floor) / (ceiling - floor), 3)

    print(f"{df['chatTime'].sample(n=1, random_state=42).iloc[0]}")
    

Normalizing game times...
CONDA_test chatTime: 2987 -> 0.834
CONDA_train chatTime: 3193 -> 0.89
CONDA_valid chatTime: 3100 -> 0.864


In [116]:
print("Removing [SEPA] Tokens...")
for name, df in conda_data.items():
    df["utterance"] = df["utterance"].str.replace(" [SEPA]", "", regex=False).str.strip()

Removing [SEPA] Tokens...


In [117]:
for name, df in conda_data.items():
    print(f"Duplicate: {name}")
    print("==================================")

    utterance_norm = df["utterance"].fillna("").str.strip()
    dupes = df.duplicated(subset=["utterance", "chatTime"], keep=False)
    longer_than_3 = utterance_norm.str.len().gt(3)

    mask = dupes & longer_than_3
    dupe_count = mask.sum()

    print(f"\nDuplicates (utterance length > 3 chars): {dupe_count}")

    if dupe_count > 0:
        print(
            df.loc[mask, ["utterance", "chatTime"]]
              .sort_values("utterance")
              .head(4)
        )

Duplicate: CONDA_test

Duplicates (utterance length > 3 chars): 59
     utterance  chatTime
3456      GGWP     0.681
2755      GGWP     1.000
8899      GGWP     0.681
3705      GGWP     1.000
Duplicate: CONDA_train

Duplicates (utterance length > 3 chars): 425
      utterance  chatTime
1054     #NAME?     0.022
19125    #NAME?     0.022
16947     GG WP     1.000
5861      GG WP     1.000
Duplicate: CONDA_valid

Duplicates (utterance length > 3 chars): 64
     utterance  chatTime
8759     all g     0.034
4972     all g     0.034
543       gege     1.000
6713      gege     0.418


In [118]:
print(f"\n[Missing Values]")
target_cols = [c for c in ["utterance", "chatTime", "intentClass"] if c in df.columns]

for col in ["utterance", "chatTime", "intentClass"]:
    if col in df.columns:
        nan_count = df[col].isna().sum()
        print(f"  {col} NaN: {nan_count}")
    else:
        print(f"  {col}: column not present")

before_rows = len(df)
if target_cols:
    df.dropna(subset=target_cols, inplace=True)
after_rows = len(df)

print(f"\n[Drop NaN Rows] dropped={before_rows - after_rows}, remaining={after_rows}")


[Missing Values]
  utterance NaN: 1
  chatTime NaN: 0
  intentClass NaN: 0

[Drop NaN Rows] dropped=1, remaining=8973


In [119]:
valid_labels = {"A","O","E","I"}

if "intentClass" in df.columns:
    unique_labels = set(df["intentClass"].dropna().unique())
    invalid = unique_labels - valid_labels

    print(f"\nLabel Validation Unique labels: {unique_labels}")
    if invalid:
        print(f"Invalid label: {invalid}")
    else:
        print(f"All labels valid")


Label Validation Unique labels: {'I', 'E', 'A', 'O'}
All labels valid


In [122]:
if "chatTime" in df.columns:
    ct = df["chatTime"]
    nan_ct = ct.isna().sum()
    out_of_range = ((ct < 0) | (ct > 1)).sum()
    print(f"\n[chatTime Range] min={ct.min()}, max={ct.max()}, NaN={nan_ct}")


[chatTime Range] min=0.0, max=1.0, NaN=0


In [121]:
print(f"Saving Cleaned Sets...\n=======================")
for name, df in conda_data.items():
    output_path = output_dir / f"{name}_cleaned.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved {name}: {df.shape} -> {output_path}")

Saving Cleaned Sets...
Saved CONDA_test: (8974, 2) -> data\processed\CONDA_test_cleaned.csv
Saved CONDA_train: (26921, 3) -> data\processed\CONDA_train_cleaned.csv
Saved CONDA_valid: (8973, 3) -> data\processed\CONDA_valid_cleaned.csv
